# Bangla Hallucination Detection — v6.0

**Goal of this revision: fix evaluation leakage in v5 and add a directly-trained signal (fine-tuned BanglaBERT) on top of the feature-engineering pipeline.**

### What changed vs v5.0

1. **Fixed leakage in blend/threshold selection.** v5 grid-searched blend weights *and* the decision threshold directly against the full OOF predictions, then reported that F1 as "the score" — that number was fit to the validation data and was optimistic. v6 reports an honest **nested-CV macro-F1** (weights/threshold selected only on training folds, evaluated on a held-out fold never used in selection) *and* a separate "deployment blend" used only to build the actual submission.
2. **Replaced the broken contradiction feature.** v5 computed "contradiction" by feeding the whole premise in as a zero-shot `candidate_label`, which is not how that pipeline is meant to be used and doesn't actually measure contradiction. v6 loads the NLI checkpoint as a proper 3-way sequence-pair classifier (entailment / neutral / contradiction) and uses the real logits — one forward pass per row instead of two abused zero-shot calls.
3. **Added a fine-tuned BanglaBERT classifier** (`csebuetnlp/banglabert`) trained directly on (premise, response) pairs with 5-fold CV. This is a *learned* signal from your actual labels, not a hand-crafted heuristic, and is usually the biggest single lever once you have a labeled training set — even a small one.
4. **Dropped LinearRegression** from the ensemble — it added a redundant, unbounded-output model into the blend search space without adding real diversity over the LR classifier.
5. **Kept** the hand-crafted features, embedding similarity, and NER features from v5, mostly unchanged.
6. Small robustness fix: test-set `id` column check restored (v2 had it, v5 assumed the column always exists).

### Honesty note

I have not run this notebook against your actual dataset (I don't have GPU/internet access to Kaggle or Hugging Face in the environment I wrote this in) — this is a code-correctness and methodology pass, not a benchmarked result. The nested-CV score printed in section 12 is the number to trust over anything printed earlier in the notebook. Whether it clears 0.70 depends on how learnable your 299-row dataset actually is; no amount of pipeline engineering can guarantee a number on data I haven't seen. If BanglaBERT overfits on your fold sizes (watch the per-fold train vs. val F1 gap in section 3), reduce epochs / increase weight decay before trusting its OOF output.

## 0. Setup

In [ ]:
!pip install -q transformers sentence-transformers xgboost accelerate
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
import json, gc, os, re, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                           get_linear_schedule_with_warmup)
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SKPipeline
import xgboost as xgb
from tqdm.auto import tqdm
from normalizer import normalize
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SUBMISSION_PATH = "/kaggle/working/submission_v6.0.csv"

NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
EMB_MODEL = "intfloat/multilingual-e5-base"
NER_MODEL = "Davlan/xlm-roberta-base-ner-hrl"
BANGLABERT_MODEL = "csebuetnlp/banglabert"

N_FOLDS = 5
MAX_LEN = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. BanglaBERT fine-tuning and the transformer feature "
          "extractors will be very slow on CPU. Set ENABLE_BANGLABERT / ENABLE_NER to False "
          "below if you need a quick run.")

ENABLE_BANGLABERT = True   # the main new signal; set False to skip if you're GPU/time constrained
ENABLE_NER = True          # slower, moderate signal; set False to skip

## 1. Load & Clean Data

In [ ]:
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../bengali-hallucination/dataset samples.json"
    TEST_PATH = "../bengali-hallucination/test set.csv"

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
df = pd.DataFrame(records)
print(f"Train: {len(df)}")

NO_CTX = {"", "nan", "NaN", "[NULL]", None}
def clean(text):
    if pd.isna(text) or str(text).strip() in NO_CTX:
        return ""
    return normalize(str(text).strip())

for col in ['context', 'prompt_bn', 'response_bn']:
    df[col] = df[col].apply(clean)

df['has_context'] = df['context'].str.len() > 0
df['premise'] = df['context'].where(df['has_context'], df['prompt_bn'])

print(f"With context: {df['has_context'].sum()}, Without: {(~df['has_context']).sum()}")
print(f"Label 0 (hallucinated): {(df['label']==0).sum()}, Label 1 (faithful): {(df['label']==1).sum()}")

# Fixed 5-fold split reused everywhere below so every OOF array lines up on the same indices
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
y = df['label'].values
fold_splits = list(skf.split(df, y))

## 2. Fine-tuned BanglaBERT Classifier (new in v6)

Trained directly on `(premise, response)` pairs with the actual competition labels — this is a learned
signal, complementary to the hand-crafted/zero-shot features below. Uses the same 5 folds as everything
else so its out-of-fold predictions can be safely used as a feature downstream (each row's prediction
comes from a model that never saw that row during training).

In [ ]:
class PairDataset(Dataset):
    def __init__(self, premises, responses, labels, tokenizer, max_len=MAX_LEN):
        self.premises = list(premises)
        self.responses = list(responses)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.premises[idx], self.responses[idx],
            truncation=True, max_length=self.max_len, padding='max_length',
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def train_one_fold(train_df, val_df, epochs=6, batch_size=8, lr=2e-5, weight_decay=0.01, patience=2):
    tokenizer = AutoTokenizer.from_pretrained(BANGLABERT_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(BANGLABERT_MODEL, num_labels=2).to(device)

    n0 = (train_df['label'] == 0).sum()
    n1 = (train_df['label'] == 1).sum()
    class_weights = torch.tensor([len(train_df) / (2 * max(n0, 1)),
                                   len(train_df) / (2 * max(n1, 1))], dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    train_ds = PairDataset(train_df['premise'], train_df['response_bn'], train_df['label'].values, tokenizer)
    val_ds = PairDataset(val_df['premise'], val_df['response_bn'], val_df['label'].values, tokenizer)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False)

    optim = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_dl) * epochs
    sched = get_linear_schedule_with_warmup(optim, num_warmup_steps=int(0.1 * total_steps),
                                             num_training_steps=total_steps)

    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            optim.zero_grad()
            out = model(**batch)
            loss = loss_fn(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            sched.step()
            train_loss += loss.item()

        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for batch in val_dl:
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop('labels')
                out = model(**batch)
                probs = F.softmax(out.logits, dim=-1)[:, 1]
                val_probs.extend(probs.cpu().numpy().tolist())
                val_labels.extend(labels.cpu().numpy().tolist())
        val_preds = (np.array(val_probs) >= 0.5).astype(int)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        print(f"    epoch {epoch+1}/{epochs}  train_loss={train_loss/len(train_dl):.4f}  val_macroF1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print("    early stopping")
                break

    model.load_state_dict(best_state)
    return model, tokenizer, best_f1


banglabert_oof = np.full(len(df), 0.5)
banglabert_fold_models = []  # kept for test-time ensembling

if ENABLE_BANGLABERT:
    for fold, (tr_idx, va_idx) in enumerate(fold_splits):
        print(f"\n--- BanglaBERT fold {fold} ---")
        tr_df, va_df = df.iloc[tr_idx], df.iloc[va_idx]
        model, tokenizer, fold_f1 = train_one_fold(tr_df, va_df)

        model.eval()
        val_ds = PairDataset(va_df['premise'], va_df['response_bn'], None, tokenizer)
        val_dl = DataLoader(val_ds, batch_size=16, shuffle=False)
        probs = []
        with torch.no_grad():
            for batch in val_dl:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(**batch)
                probs.extend(F.softmax(out.logits, dim=-1)[:, 1].cpu().numpy().tolist())
        banglabert_oof[va_idx] = probs
        banglabert_fold_models.append((model, tokenizer))
        del model
        gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

    df['banglabert_oof'] = banglabert_oof
    oof_acc = accuracy_score(y, (banglabert_oof >= 0.5).astype(int))
    oof_f1 = f1_score(y, (banglabert_oof >= 0.5).astype(int), average='macro')
    print(f"\n=== BanglaBERT OOF: Acc={oof_acc:.4f}  MacroF1={oof_f1:.4f} ===")
else:
    df['banglabert_oof'] = 0.5
    print("BanglaBERT disabled — feature set to neutral 0.5 for all rows.")

## 3. Hand-Crafted Features (kept from v5)

In [ ]:
BN_DIGITS = {'০':0,'১':1,'২':2,'৩':3,'৪':4,'৫':5,'৬':6,'৭':7,'৮':8,'৯':9}

def bn_to_int(s):
    try:
        return int(s)
    except Exception:
        pass
    try:
        s2 = s
        for bn, ar in BN_DIGITS.items():
            s2 = s2.replace(bn, str(ar))
        return int(s2)
    except Exception:
        return None

def extract_numbers(text):
    nums = re.findall(r'[০-৯]+|[0-9]+(?:\.[0-9]+)?', text)
    result = []
    for n in nums:
        v = bn_to_int(n)
        if v is not None:
            result.append(v)
    return result

def tokenize_bangla(text):
    return [w for w in re.split(r'[\s,\.\?\!\?\"\'\:\;\(\)\[\]\{\}]+', text) if len(w) > 0]

def jaccard(a, b):
    if not a or not b:
        return 0.0
    sa, sb = set(a), set(b)
    return len(sa & sb) / max(len(sa | sb), 1)

def common_ratio(a, b):
    if not b:
        return 0.0
    return len(set(a) & set(b)) / max(len(set(b)), 1)

def build_features(data_df):
    feats = pd.DataFrame(index=data_df.index)
    feats['ctx_len'] = data_df['context'].str.len()
    feats['prompt_len'] = data_df['prompt_bn'].str.len()
    feats['resp_len'] = data_df['response_bn'].str.len()
    feats['has_context'] = data_df['has_context'].astype(int)
    feats['resp_prompt_ratio'] = feats['resp_len'] / (feats['prompt_len'] + 1)
    feats['resp_ctx_ratio'] = feats['resp_len'] / (feats['ctx_len'] + 1)
    feats['ctx_prompt_ratio'] = feats['ctx_len'] / (feats['prompt_len'] + 1)

    prompt_toks = data_df['prompt_bn'].apply(tokenize_bangla)
    resp_toks = data_df['response_bn'].apply(tokenize_bangla)
    ctx_toks = data_df['context'].apply(tokenize_bangla)

    feats['prompt_n_tokens'] = prompt_toks.apply(len)
    feats['resp_n_tokens'] = resp_toks.apply(len)
    feats['ctx_n_tokens'] = ctx_toks.apply(len)

    feats['resp_in_ctx_jaccard'] = [jaccard(r, c) for r, c in zip(resp_toks, ctx_toks)]
    feats['resp_in_ctx_ratio'] = [common_ratio(r, c) for r, c in zip(resp_toks, ctx_toks)]
    feats['ctx_in_resp_ratio'] = [common_ratio(c, r) for r, c in zip(resp_toks, ctx_toks)]
    feats['resp_prompt_jaccard'] = [jaccard(r, p) for r, p in zip(resp_toks, prompt_toks)]
    feats['resp_in_prompt_ratio'] = [common_ratio(r, p) for r, p in zip(resp_toks, prompt_toks)]

    ctx_nums = data_df['context'].apply(extract_numbers)
    resp_nums = data_df['response_bn'].apply(extract_numbers)
    prompt_nums = data_df['prompt_bn'].apply(extract_numbers)

    feats['resp_has_number'] = resp_nums.apply(len).clip(upper=1)
    feats['ctx_has_number'] = ctx_nums.apply(len).clip(upper=1)
    feats['prompt_has_number'] = prompt_nums.apply(len).clip(upper=1)

    def numbers_match(rn, cn):
        if not rn:
            return 0
        return int(any(n in cn for n in rn))

    feats['resp_num_in_ctx'] = [numbers_match(rn, cn) for rn, cn in zip(resp_nums, ctx_nums)]
    feats['resp_num_count'] = resp_nums.apply(len)
    feats['ctx_num_count'] = ctx_nums.apply(len)
    feats['prompt_num_count'] = prompt_nums.apply(len)

    feats['resp_very_short'] = (feats['resp_len'] < 15).astype(int)
    feats['resp_one_word'] = (feats['resp_n_tokens'] <= 2).astype(int)

    feats['resp_starts_with_ctx'] = [
        int(str(r).startswith(str(c)[:20])) if c and r else 0
        for r, c in zip(data_df['response_bn'], data_df['context'])
    ]

    feats['resp_extrinsic_words'] = [len(set(r) - set(c)) if c else 0 for r, c in zip(resp_toks, ctx_toks)]
    feats['resp_extrinsic_ratio'] = [
        len(set(r) - set(c)) / max(len(set(r)), 1) if c else 1.0 for r, c in zip(resp_toks, ctx_toks)
    ]
    feats['ctx_resp_shared_entities'] = [len(set(r) & set(c)) if c else 0 for r, c in zip(resp_toks, ctx_toks)]
    feats['ctx_resp_entity_overlap'] = [
        len(set(r) & set(c)) / max(len(set(r) | set(c)), 1) if c else 0 for r, c in zip(resp_toks, ctx_toks)
    ]
    feats['prompt_resp_shared_entities'] = [len(set(p) & set(r)) for p, r in zip(prompt_toks, resp_toks)]
    feats['prompt_resp_entity_overlap'] = [
        len(set(p) & set(r)) / max(len(set(p) | set(r)), 1) for p, r in zip(prompt_toks, resp_toks)
    ]
    feats['ctx_completeness'] = [
        len(set(c) & set(p)) / max(len(set(p)), 1) if c else 0 for c, p in zip(ctx_toks, prompt_toks)
    ]
    return feats.fillna(0)

print("Building hand-crafted features...")
hc_train = build_features(df)
print(f"Hand-crafted features: {hc_train.shape[1]} columns")

## 4. Proper 3-Way NLI Features (fixed in v6)

The v5 "contradiction" feature abused the zero-shot pipeline by passing a whole premise in as a
`candidate_label`. Here we load the same checkpoint as an ordinary NLI sequence-pair classifier and take
its real entailment / neutral / contradiction logits directly — one forward pass per row instead of two
zero-shot calls, and the numbers actually mean what they say.

In [ ]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
nli_model.eval()
# MoritzLaurer's checkpoint label order is entailment=0, neutral=1, contradiction=2
NLI_ENTAIL_IDX, NLI_NEUTRAL_IDX, NLI_CONTRA_IDX = 0, 1, 2

@torch.no_grad()
def get_nli_scores(data_df, batch_size=16):
    premises = data_df['premise'].tolist()
    hypotheses = data_df['response_bn'].tolist()
    entail = np.full(len(data_df), 1/3)
    neutral = np.full(len(data_df), 1/3)
    contra = np.full(len(data_df), 1/3)

    for start in tqdm(range(0, len(data_df), batch_size), desc='NLI'):
        end = start + batch_size
        p_batch = premises[start:end]
        h_batch = hypotheses[start:end]
        valid = [i for i, (p, h) in enumerate(zip(p_batch, h_batch)) if p and h]
        if not valid:
            continue
        enc = nli_tokenizer([p_batch[i] for i in valid], [h_batch[i] for i in valid],
                             truncation=True, max_length=384, padding=True, return_tensors='pt').to(device)
        logits = nli_model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        for j, i in enumerate(valid):
            entail[start + i] = probs[j, NLI_ENTAIL_IDX]
            neutral[start + i] = probs[j, NLI_NEUTRAL_IDX]
            contra[start + i] = probs[j, NLI_CONTRA_IDX]
    return entail, neutral, contra

print("Computing NLI scores for training data...")
nli_entail, nli_neutral, nli_contra = get_nli_scores(df)
df['nli_entail'] = nli_entail
df['nli_neutral'] = nli_neutral
df['nli_contra'] = nli_contra
print(f"NLI entail: mean={nli_entail.mean():.3f}  neutral: mean={nli_neutral.mean():.3f}  contra: mean={nli_contra.mean():.3f}")

ctx_mask = df['has_context']
ctx_acc = accuracy_score(df.loc[ctx_mask, 'label'], (nli_entail[ctx_mask] > 0.5).astype(int))
print(f"NLI-entailment-only accuracy on context rows: {ctx_acc:.4f}")

del nli_model
gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

## 5. Embedding Similarity (multilingual-e5-base, kept from v5)

In [ ]:
emb_model = SentenceTransformer(EMB_MODEL, device=str(device))

def cosine_sim_matrix(texts_a, texts_b, batch_size=64):
    emb_a = emb_model.encode(texts_a, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    emb_b = emb_model.encode(texts_b, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    sims = np.sum(emb_a * emb_b, axis=1)
    return sims, emb_a, emb_b

print("Encoding training texts...")
ctx_texts = df['premise'].tolist()
resp_texts = df['response_bn'].tolist()
prompt_texts = df['prompt_bn'].tolist()

sim_ctx_resp, emb_ctx, emb_resp = cosine_sim_matrix(ctx_texts, resp_texts)
sim_prompt_resp, _, _ = cosine_sim_matrix(prompt_texts, resp_texts)

df['sim_ctx_resp'] = sim_ctx_resp
df['sim_prompt_resp'] = sim_prompt_resp
print(f"sim_ctx_resp: mean={sim_ctx_resp.mean():.3f}  sim_prompt_resp: mean={sim_prompt_resp.mean():.3f}")

del emb_model
gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

## 6. NER Features (kept from v5, optional)

In [ ]:
def get_ner_features(data_df):
    ner_feats = pd.DataFrame(index=data_df.index)
    if not ENABLE_NER:
        for col in ['ctx_entity_count', 'resp_entity_count', 'prompt_entity_count',
                    'resp_ctx_entity_overlap', 'resp_prompt_entity_overlap',
                    'resp_entities_in_ctx', 'resp_entities_in_prompt',
                    'resp_extrinsic_entities', 'resp_extrinsic_entity_ratio']:
            ner_feats[col] = 0.0
        return ner_feats

    from transformers import pipeline as hf_pipeline
    ner_pipe = hf_pipeline("ner", model=NER_MODEL, device=0 if device == 'cuda' else -1,
                            aggregation_strategy="simple")

    def extract_entities(text):
        if not text or len(text.strip()) == 0:
            return []
        try:
            return [e['word'] for e in ner_pipe(text)]
        except Exception:
            return []

    ctx_entities = data_df['context'].apply(extract_entities)
    resp_entities = data_df['response_bn'].apply(extract_entities)
    prompt_entities = data_df['prompt_bn'].apply(extract_entities)

    ner_feats['ctx_entity_count'] = ctx_entities.apply(len)
    ner_feats['resp_entity_count'] = resp_entities.apply(len)
    ner_feats['prompt_entity_count'] = prompt_entities.apply(len)
    ner_feats['resp_ctx_entity_overlap'] = [
        len(set(r) & set(c)) / max(len(set(r) | set(c)), 1) if c else 0
        for r, c in zip(resp_entities, ctx_entities)
    ]
    ner_feats['resp_prompt_entity_overlap'] = [
        len(set(r) & set(p)) / max(len(set(r) | set(p)), 1)
        for r, p in zip(resp_entities, prompt_entities)
    ]
    ner_feats['resp_entities_in_ctx'] = [
        len(set(r) & set(c)) / max(len(set(r)), 1) if c else 0
        for r, c in zip(resp_entities, ctx_entities)
    ]
    ner_feats['resp_entities_in_prompt'] = [
        len(set(r) & set(p)) / max(len(set(r)), 1)
        for r, p in zip(resp_entities, prompt_entities)
    ]
    ner_feats['resp_extrinsic_entities'] = [
        len(set(r) - set(c)) if c else len(set(r)) for r, c in zip(resp_entities, ctx_entities)
    ]
    ner_feats['resp_extrinsic_entity_ratio'] = [
        len(set(r) - set(c)) / max(len(set(r)), 1) if c else 1.0
        for r, c in zip(resp_entities, ctx_entities)
    ]
    del ner_pipe
    gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None
    return ner_feats.fillna(0)

print("Computing NER features for training data...")
ner_train = get_ner_features(df)
print(f"NER features: {ner_train.shape[1]} columns")

## 7. Assemble Full Feature Matrix

In [ ]:
feature_cols = (
    list(hc_train.columns) +
    list(ner_train.columns) +
    ['nli_entail', 'nli_neutral', 'nli_contra', 'sim_ctx_resp', 'sim_prompt_resp', 'banglabert_oof']
)

X_full = np.column_stack([
    hc_train.values,
    ner_train.values,
    df['nli_entail'].values.reshape(-1, 1),
    df['nli_neutral'].values.reshape(-1, 1),
    df['nli_contra'].values.reshape(-1, 1),
    df['sim_ctx_resp'].values.reshape(-1, 1),
    df['sim_prompt_resp'].values.reshape(-1, 1),
    df['banglabert_oof'].values.reshape(-1, 1),
])

print(f"Feature matrix: {X_full.shape}")
print(f"Features: {len(feature_cols)} columns -> {feature_cols}")
print(f"Label distribution: 0={(y==0).sum()}, 1={(y==1).sum()}")

## 8. 5-Fold CV: Logistic Regression + XGBoost

(LinearRegression dropped — it was redundant with LR and only added blend-search surface area.)

In [ ]:
oof_lr = np.zeros(len(df))
oof_xgb = np.zeros(len(df))
n0, n1 = (y == 0).sum(), (y == 1).sum()

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f"\n--- Fold {fold} ---")
    X_tr, X_va = X_full[tr_idx], X_full[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    lr = SKPipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=0.5, max_iter=1000, class_weight='balanced',
                                    penalty='l2', random_state=RANDOM_STATE)),
    ])
    lr.fit(X_tr, y_tr)
    oof_lr[va_idx] = lr.predict_proba(X_va)[:, 1]

    xgb_clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        reg_lambda=2.0, scale_pos_weight=n0 / n1, random_state=RANDOM_STATE,
        eval_metric='logloss', early_stopping_rounds=20, verbosity=0
    )
    xgb_clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx] = xgb_clf.predict_proba(X_va)[:, 1]

    print(f"  LR   Acc={accuracy_score(y_va, (oof_lr[va_idx]>=0.5).astype(int)):.4f}  "
          f"F1={f1_score(y_va, (oof_lr[va_idx]>=0.5).astype(int), average='macro'):.4f}")
    print(f"  XGB  Acc={accuracy_score(y_va, (oof_xgb[va_idx]>=0.5).astype(int)):.4f}  "
          f"F1={f1_score(y_va, (oof_xgb[va_idx]>=0.5).astype(int), average='macro'):.4f}")

print(f"\n=== LR  OOF: Acc={accuracy_score(y,(oof_lr>=0.5).astype(int)):.4f}  F1={f1_score(y,(oof_lr>=0.5).astype(int),average='macro'):.4f}")
print(f"=== XGB OOF: Acc={accuracy_score(y,(oof_xgb>=0.5).astype(int)):.4f}  F1={f1_score(y,(oof_xgb>=0.5).astype(int),average='macro'):.4f}")
print(f"=== BanglaBERT OOF: Acc={accuracy_score(y,(banglabert_oof>=0.5).astype(int)):.4f}  F1={f1_score(y,(banglabert_oof>=0.5).astype(int),average='macro'):.4f}")

## 9. Blend weight search helper

Same 4-component grid search as v5 (LR, XGB, NLI-entail, embedding-sim — BanglaBERT is already folded
into X_full and therefore into LR/XGB, but we also let it compete directly in the blend since it can be
a stronger standalone signal than the tree/linear models built on top of it).

In [ ]:
nli_raw = df['nli_entail'].values
sim_raw = df['sim_ctx_resp'].values
bb_raw = df['banglabert_oof'].values

OOF_COMPONENTS = {
    'lr': oof_lr, 'xgb': oof_xgb, 'nli': nli_raw, 'sim': sim_raw, 'bb': bb_raw,
}

def grid_search_blend(oof_components, y_subset, idx_subset, step=0.05, thresholds=np.arange(0.30, 0.71, 0.01)):
    """Grid search blend weights (summing to 1) and threshold on the given subset of indices only."""
    names = list(oof_components.keys())
    arrs = [oof_components[n][idx_subset] for n in names]
    y_sub = y_subset

    best_f1, best_w, best_t = -1, None, 0.5
    # coarse weight grid over 5 components, constrained to sum to 1
    grid_vals = np.arange(0, 1.01, step)
    def rec(k, remaining, chosen):
        nonlocal best_f1, best_w, best_t
        if k == len(names) - 1:
            w = chosen + [remaining]
            if remaining < -1e-6:
                return
            blend = sum(wi * ai for wi, ai in zip(w, arrs))
            for t in thresholds:
                p = (blend >= t).astype(int)
                f1 = f1_score(y_sub, p, average='macro')
                if f1 > best_f1:
                    best_f1, best_w, best_t = f1, tuple(w), t
            return
        for wv in grid_vals:
            if wv > remaining + 1e-6:
                break
            rec(k + 1, round(remaining - wv, 4), chosen + [wv])
    rec(0, 1.0, [])
    return dict(zip(names, best_w)), best_t, best_f1

## 10. Honest Nested-CV Score (this is the number to trust)

For each outer fold, the blend weights and threshold are selected **using only the other 4 folds**, then
applied — unseen — to the held-out fold. This is what v5 skipped; it's the difference between a number
that's fit to your validation data and a number that estimates real generalization.

In [ ]:
outer_fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    weights, thresh, inner_f1 = grid_search_blend(OOF_COMPONENTS, y[tr_idx], tr_idx)
    blend_va = sum(w * OOF_COMPONENTS[name][va_idx] for name, w in weights.items())
    preds_va = (blend_va >= thresh).astype(int)
    fold_f1 = f1_score(y[va_idx], preds_va, average='macro')
    outer_fold_scores.append(fold_f1)
    print(f"Outer fold {fold}: inner-selected weights={ {k: round(v,2) for k,v in weights.items()} }  "
          f"thresh={thresh:.2f}  ->  held-out MacroF1={fold_f1:.4f}")

honest_cv_score = np.mean(outer_fold_scores)
honest_cv_std = np.std(outer_fold_scores)
print(f"\n=== Honest nested-CV Macro-F1: {honest_cv_score:.4f} (+/- {honest_cv_std:.4f}) ===")
print("This is the number that should predict your leaderboard score, not any of the earlier ones.")

## 11. Deployment Blend (for generating the actual submission)

Now that we have an honest estimate, do the full-data grid search to pick the weights/threshold that will
actually be used at inference time. This number will look better than section 10 — that's expected and
is *not* the number to report to yourself as "my score."

In [ ]:
all_idx = np.arange(len(df))
deploy_weights, deploy_thresh, deploy_f1 = grid_search_blend(OOF_COMPONENTS, y, all_idx)
print(f"Deployment weights: { {k: round(v,2) for k,v in deploy_weights.items()} }")
print(f"Deployment threshold: {deploy_thresh:.2f}")
print(f"(In-sample OOF Macro-F1 with these weights: {deploy_f1:.4f} — optimistic, see section 10 for the honest estimate)")

final_blend = sum(w * OOF_COMPONENTS[name] for name, w in deploy_weights.items())
final_preds = (final_blend >= deploy_thresh).astype(int)
print(confusion_matrix(y, final_preds))
print(classification_report(y, final_preds, target_names=['hallucinated(0)', 'faithful(1)']))

## 12. Test Set Inference

In [ ]:
if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)
    print(f"Test: {len(test_df)}")

    for col in ['context', 'prompt_bn', 'response_bn']:
        test_df[col] = test_df[col].apply(clean)
    test_df['has_context'] = test_df['context'].str.len() > 0
    test_df['premise'] = test_df['context'].where(test_df['has_context'], test_df['prompt_bn'])

    # --- BanglaBERT: average predictions across the 5 fold models (no single fold saw all training data,
    #     this ensembling avoids having to pick just one, and tends to be more stable than a full retrain) ---
    if ENABLE_BANGLABERT:
        test_bb_probs = np.zeros(len(test_df))
        for model, tokenizer in banglabert_fold_models:
            model.eval()
            ds = PairDataset(test_df['premise'], test_df['response_bn'], None, tokenizer)
            dl = DataLoader(ds, batch_size=16, shuffle=False)
            fold_probs = []
            with torch.no_grad():
                for batch in dl:
                    batch = {k: v.to(device) for k, v in batch.items()}
                    out = model(**batch)
                    fold_probs.extend(F.softmax(out.logits, dim=-1)[:, 1].cpu().numpy().tolist())
            test_bb_probs += np.array(fold_probs) / len(banglabert_fold_models)
        test_df['banglabert_oof'] = test_bb_probs
    else:
        test_df['banglabert_oof'] = 0.5

    # --- NLI ---
    print("Computing test NLI scores...")
    nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
    nli_model.eval()
    test_nli_entail, test_nli_neutral, test_nli_contra = get_nli_scores(test_df)
    test_df['nli_entail'] = test_nli_entail
    test_df['nli_neutral'] = test_nli_neutral
    test_df['nli_contra'] = test_nli_contra
    del nli_model; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

    # --- Embeddings ---
    print("Computing test embedding similarity...")
    emb_model = SentenceTransformer(EMB_MODEL, device=str(device))
    test_ctx_texts = test_df['premise'].tolist()
    test_resp_texts = test_df['response_bn'].tolist()
    test_prompt_texts = test_df['prompt_bn'].tolist()
    test_sim_ctx_resp, _, _ = cosine_sim_matrix(test_ctx_texts, test_resp_texts)
    test_sim_prompt_resp, _, _ = cosine_sim_matrix(test_prompt_texts, test_resp_texts)
    test_df['sim_ctx_resp'] = test_sim_ctx_resp
    test_df['sim_prompt_resp'] = test_sim_prompt_resp
    del emb_model; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

    # --- NER + hand-crafted ---
    print("Computing test NER + hand-crafted features...")
    ner_test = get_ner_features(test_df)
    hc_test = build_features(test_df)

    X_test = np.column_stack([
        hc_test.values,
        ner_test.values,
        test_df['nli_entail'].values.reshape(-1, 1),
        test_df['nli_neutral'].values.reshape(-1, 1),
        test_df['nli_contra'].values.reshape(-1, 1),
        test_df['sim_ctx_resp'].values.reshape(-1, 1),
        test_df['sim_prompt_resp'].values.reshape(-1, 1),
        test_df['banglabert_oof'].values.reshape(-1, 1),
    ])
    print(f"Test features: {X_test.shape}")

    # --- Retrain LR / XGB on full training data for test-time inference ---
    lr_full = SKPipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=0.5, max_iter=1000, class_weight='balanced',
                                    penalty='l2', random_state=RANDOM_STATE)),
    ])
    lr_full.fit(X_full, y)
    lr_test_prob = lr_full.predict_proba(X_test)[:, 1]

    xgb_full = xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        reg_lambda=2.0, scale_pos_weight=n0 / n1, random_state=RANDOM_STATE,
        eval_metric='logloss', verbosity=0
    )
    xgb_full.fit(X_full, y)
    xgb_test_prob = xgb_full.predict_proba(X_test)[:, 1]

    test_components = {
        'lr': lr_test_prob, 'xgb': xgb_test_prob,
        'nli': test_nli_entail, 'sim': test_sim_ctx_resp, 'bb': test_df['banglabert_oof'].values,
    }
    test_blend = sum(deploy_weights[name] * arr for name, arr in test_components.items())
    test_preds = (test_blend >= deploy_thresh).astype(int)

    n0_pred, n1_pred = (test_preds == 0).sum(), (test_preds == 1).sum()
    print(f"Test distribution: 0={n0_pred} ({n0_pred/len(test_preds)*100:.1f}%), "
          f"1={n1_pred} ({n1_pred/len(test_preds)*100:.1f}%)")

    id_col = test_df['id'] if 'id' in test_df.columns else range(len(test_df))
    sub = pd.DataFrame({'id': id_col, 'label': test_preds})
    os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
    sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"Saved: {SUBMISSION_PATH}")
    sub.head(10)
else:
    print("Test set CSV file not found. Skipping prediction. Ready for Kaggle run.")

## Summary

**v6.0 changes vs v5.0:**
- Fixed evaluation leakage: blend-weight and threshold search now uses nested CV (section 10) for the
  score you should trust, separate from the deployment blend (section 11) used to build the submission.
- Replaced the misused zero-shot "contradiction" hack with a real 3-way NLI classification (entailment /
  neutral / contradiction) from a single correct forward pass.
- Added a fine-tuned BanglaBERT classifier trained directly on your labels via 5-fold CV — this is the
  main new lever for pushing the score, since it's the only component that learns directly from the
  task rather than relying on hand-crafted heuristics or off-the-shelf zero-shot behavior.
- Dropped LinearRegression from the ensemble (redundant with LR, added search space without diversity).
- Restored the defensive `id` column check on the test set.
- Kept hand-crafted features, embedding similarity, and NER features from v5 largely unchanged.

**What I'd watch for when you actually run this:**
- If BanglaBERT's per-fold train/val F1 gap is large (printed in section 2), it's overfitting on your
  fold size — cut epochs, raise weight decay, or freeze the lower encoder layers.
- If `honest_cv_score` (section 10) is noticeably lower than the deployment F1 (section 11), that gap
  itself tells you how much the old approach was overstating performance — trust section 10.
- With ~299 rows, expect some fold-to-fold variance in section 10's per-fold scores; the standard
  deviation printed there is as important as the mean.
- I could not execute this against your actual dataset/GPU in the environment I wrote it in, so treat
  this as a correctness/methodology pass — please run it on Kaggle and share the section 10 output if
  you want help interpreting or iterating further.